In [1]:
import pyarrow.parquet as pq

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAP.parquet'
pf = pq.ParquetFile(path)

total_rows = pf.metadata.num_rows
null_counts = {col: 0 for col in pf.schema_arrow.names}

# Read in manageable batches
for batch in pf.iter_batches(batch_size=100_000):
    for col in batch.schema.names:
        null_counts[col] += batch.column(col).null_count

print("=" * 80)
print(f"AZT1D — EMPTY VALUES PER COLUMN")
print(f"Total rows: {total_rows:,}")
print("=" * 80)

results = []

for col, null_count in null_counts.items():
    percentage = (null_count / total_rows) * 100

    results.append((col, null_count, percentage))

# Sort from most empty to least empty
results.sort(key=lambda x: x[2], reverse=True)

for col, null_count, percentage in results:
    print(f"{col:<40} {null_count:>12,} empty | {percentage:>8.2f}%")

AZT1D — EMPTY VALUES PER COLUMN
Total rows: 5,507,753
basal                                       2,559,928 empty |    46.48%
CGM                                           518,062 empty |     9.41%
height                                        204,810 empty |     3.72%
carbs                                           9,081 empty |     0.16%
bolus                                           5,598 empty |     0.10%
insulin                                         4,189 empty |     0.08%
age                                                 0 empty |     0.00%
cgm_device                                          0 empty |     0.00%
date                                                0 empty |     0.00%
id                                                  0 empty |     0.00%
insulin_delivery_algorithm                          0 empty |     0.00%
insulin_delivery_device                             0 empty |     0.00%
insulin_delivery_modality                           0 empty |     0.00%
insulin_ty

In [1]:
import pyarrow.parquet as pq

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAP.parquet'

pf = pq.ParquetFile(path)

# Read only the first 10 rows
table = pf.iter_batches(batch_size=300).__next__()
df = table.to_pandas()

print(df.to_string(index=False))

  CGM  age    basal  bolus  carbs cgm_device                date   height id  insulin insulin_delivery_algorithm insulin_delivery_device insulin_delivery_modality insulin_type_basal insulin_type_bolus  is_pregnant  is_test source_file    weight gender  age_of_diagnosis ethnicity  subject_split_across_traintest patient_stream_id
  NaN  4.0 0.025000 1.1100   22.0  Dexcom G6 2020-12-28 13:30:00 3.425197 27 1.135000                 Control-IQ               t:slim X2                       AID   Humalog (Lispro)   Humalog (Lispro)        False    False       PEDAP 43.210552 Female               3.0     White                           False          PEDAP_27
  NaN  4.0 0.025000 0.0000    0.0  Dexcom G6 2020-12-28 13:35:00 3.425197 27 0.025000                 Control-IQ               t:slim X2                       AID   Humalog (Lispro)   Humalog (Lispro)        False    False       PEDAP 43.210552 Female               3.0     White                           False          PEDAP_27
  NaN  4.0

In [1]:
import pyarrow.parquet as pq
from collections import Counter

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAP.parquet'
pf = pq.ParquetFile(path)

values = Counter()

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["insulin_type_bolus"]
):
    for value in batch.column("insulin_type_bolus").to_pylist():
        if value is not None:
            values[value] += 1

print("Number of different values:", len(values))

print("\nDifferent insulin_type_bolus values:")
for value, count in values.most_common():
    print(f"{value}: {count}")

Number of different values: 3

Different insulin_type_bolus values:
Humalog (Lispro): 3202549
Novolog (Aspart): 2116369
Humalog (Lispro) or Novolog (Aspart): 188835


In [2]:
import pyarrow.parquet as pq

path = "/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAP.parquet"

pf = pq.ParquetFile(path)

uncertain_value = "Humalog (Lispro) or Novolog (Aspart)"

subjects = set()
total_uncertain = 0

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["id", "insulin_type_bolus"]
):
    ids = batch.column("id").to_pylist()
    types = batch.column("insulin_type_bolus").to_pylist()

    for subject_id, insulin_type in zip(ids, types):
        if insulin_type == uncertain_value:
            subjects.add(subject_id)
            total_uncertain += 1

print("=" * 70)
print("UNCERTAIN INSULIN TYPE")
print("=" * 70)

print("Total uncertain rows:", total_uncertain)
print("Number of subjects:", len(subjects))

print("\nSubjects:")
for subject in sorted(subjects, key=str):
    print(subject)

UNCERTAIN INSULIN TYPE
Total uncertain rows: 188835
Number of subjects: 2

Subjects:
23
84


In [4]:
import pyarrow.parquet as pq

path = "/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAP_backup.parquet"

pf = pq.ParquetFile(path)

print(pf.schema.names)

['CGM', 'age', 'basal', 'bolus', 'carbs', 'cgm_device', 'date', 'height', 'id', 'insulin', 'insulin_delivery_algorithm', 'insulin_delivery_device', 'insulin_delivery_modality', 'insulin_type_basal', 'insulin_type_bolus', 'is_pregnant', 'is_test', 'source_file', 'weight', 'gender', 'age_of_diagnosis', 'ethnicity', 'subject_split_across_traintest', 'patient_stream_id']


In [5]:
import pandas as pd

table = pf.read_row_group(0)
df = table.to_pandas()

print(df.head())
print("\nColumns:")
print(df.columns.tolist())

   CGM  age  basal  bolus  carbs cgm_device                date    height  id  \
0  NaN  4.0  0.025   1.11   22.0  Dexcom G6 2020-12-28 13:30:00  3.425197  27   
1  NaN  4.0  0.025   0.00    0.0  Dexcom G6 2020-12-28 13:35:00  3.425197  27   
2  NaN  4.0  0.025   0.00    0.0  Dexcom G6 2020-12-28 13:40:00  3.425197  27   
3  NaN  4.0  0.025   0.00    0.0  Dexcom G6 2020-12-28 13:45:00  3.425197  27   
4  NaN  4.0  0.025   0.00    0.0  Dexcom G6 2020-12-28 13:50:00  3.425197  27   

   insulin  ... insulin_type_bolus is_pregnant is_test source_file     weight  \
0    1.135  ...   Humalog (Lispro)       False   False       PEDAP  43.210552   
1    0.025  ...   Humalog (Lispro)       False   False       PEDAP  43.210552   
2    0.025  ...   Humalog (Lispro)       False   False       PEDAP  43.210552   
3    0.025  ...   Humalog (Lispro)       False   False       PEDAP  43.210552   
4    0.025  ...   Humalog (Lispro)       False   False       PEDAP  43.210552   

   gender  age_of_diagnosi

In [6]:
import pyarrow.parquet as pq

path = "/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAP_backup.parquet"

pf = pq.ParquetFile(path)

target = "Humalog (Lispro) or Novolog (Aspart)"

found = []

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["id", "patient_stream_id", "insulin_type_bolus"]
):
    df = batch.to_pandas()

    matches = df[df["insulin_type_bolus"] == target]

    if len(matches) > 0:
        found.append(
            matches[["id", "patient_stream_id", "insulin_type_bolus"]]
        )

result = pd.concat(found, ignore_index=True)

print("Rows:", len(result))
print("\nSubjects:")
print(result["id"].value_counts())

print("\nPatient streams:")
print(result["patient_stream_id"].value_counts())

Rows: 188835

Subjects:
id
84    100764
23     88071
Name: count, dtype: int64

Patient streams:
patient_stream_id
PEDAP_84    100764
PEDAP_23     88071
Name: count, dtype: int64


In [ ]:
#This script creates PEDAP.parquet from PEDAP_backup.parquet, removing every row whose insulin_type_bolus is exactly:
#Humalog (Lispro) or Novolog (Aspart)
#insulin_type_bolus	Keep?
#Humalog (Lispro) or Novolog (Aspart)	No
#Humalog (Lispro)	                    Yes
#None / null	                        Yes
import pyarrow.parquet as pq
import os

input_path = "/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAP_backup.parquet"
output_path = "/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAP.parquet"
temp_path = "/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAP_clean_temp.parquet"

target = "Humalog (Lispro) or Novolog (Aspart)"

pf = pq.ParquetFile(input_path)

writer = None
removed_rows = 0
kept_rows = 0

for i in range(pf.num_row_groups):

    table = pf.read_row_group(i)

    insulin_types = table.column("insulin_type_bolus").to_pylist()

    # Keep everything except the ambiguous insulin type
    keep_mask = [
        value != target
        for value in insulin_types
    ]

    removed = len(keep_mask) - sum(keep_mask)

    removed_rows += removed
    kept_rows += sum(keep_mask)

    filtered_table = table.filter(keep_mask)

    if filtered_table.num_rows > 0:

        if writer is None:
            writer = pq.ParquetWriter(
                temp_path,
                filtered_table.schema,
                compression="snappy"
            )

        writer.write_table(filtered_table)

    print(
        f"Row group {i + 1}/{pf.num_row_groups} | "
        f"Removed: {removed:,}"
    )

writer.close()

# Replace current file
os.replace(temp_path, output_path)

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)
print(f"Rows removed: {removed_rows:,}")
print(f"Rows kept:    {kept_rows:,}")
print(f"Output:       {output_path}")

Row group 1/56 | Removed: 0
Row group 2/56 | Removed: 0
Row group 3/56 | Removed: 0
Row group 4/56 | Removed: 0
Row group 5/56 | Removed: 0
Row group 6/56 | Removed: 0
Row group 7/56 | Removed: 0
Row group 8/56 | Removed: 0
Row group 9/56 | Removed: 0
Row group 10/56 | Removed: 0
Row group 11/56 | Removed: 0
Row group 12/56 | Removed: 0
Row group 13/56 | Removed: 0
Row group 14/56 | Removed: 0
Row group 15/56 | Removed: 0
Row group 16/56 | Removed: 0
Row group 17/56 | Removed: 0
Row group 18/56 | Removed: 0
Row group 19/56 | Removed: 0
Row group 20/56 | Removed: 0
Row group 21/56 | Removed: 0
Row group 22/56 | Removed: 0
Row group 23/56 | Removed: 0
Row group 24/56 | Removed: 0
Row group 25/56 | Removed: 7,645
Row group 26/56 | Removed: 93,119
Row group 27/56 | Removed: 0
Row group 28/56 | Removed: 0
Row group 29/56 | Removed: 0
Row group 30/56 | Removed: 0
Row group 31/56 | Removed: 0
Row group 32/56 | Removed: 0
Row group 33/56 | Removed: 0
Row group 34/56 | Removed: 0
Row group 35/5

In [12]:
import pyarrow.parquet as pq

path = "/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAP.parquet"

pf = pq.ParquetFile(path)

count = 0

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["insulin_type_bolus"]
):
    values = batch.column("insulin_type_bolus").to_pylist()
    count += sum(
        x == "Humalog (Lispro) or Novolog (Aspart)"
        for x in values
    )

print("Ambiguous rows:", count)
print("Total rows:", pf.metadata.num_rows)

Ambiguous rows: 0
Total rows: 5318918


In [13]:
import pyarrow.parquet as pq
from collections import Counter

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAP.parquet'
pf = pq.ParquetFile(path)

values = Counter()

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["insulin_type_bolus"]
):
    for value in batch.column("insulin_type_bolus").to_pylist():
        if value is not None:
            values[value] += 1

print("Number of different values:", len(values))

print("\nDifferent insulin_type_bolus values:")
for value, count in values.most_common():
    print(f"{value}: {count}")

Number of different values: 2

Different insulin_type_bolus values:
Humalog (Lispro): 3202549
Novolog (Aspart): 2116369


In [1]:
import pyarrow.parquet as pq
import pandas as pd

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAP.parquet'


pf = pq.ParquetFile(path)

nan_counts = {col: 0 for col in pf.schema.names}
total_rows = pf.metadata.num_rows

for batch in pf.iter_batches(batch_size=100_000):
    df = batch.to_pandas()

    for col in df.columns:
        nan_counts[col] += df[col].isna().sum()

result = pd.DataFrame({
    "Column": nan_counts.keys(),
    "NaN Count": nan_counts.values(),
})

result["NaN Percentage"] = (
    result["NaN Count"] / total_rows * 100
)

result = result.sort_values("NaN Percentage", ascending=False)

print(f"Total rows: {total_rows:,}")
print("=" * 80)
print(result.to_string(index=False))

Total rows: 5,318,918
                        Column  NaN Count  NaN Percentage
                         basal    2423155       45.557292
                           CGM     503462        9.465497
                        height     204810        3.850595
                         carbs       8987        0.168963
                         bolus       5506        0.103517
                       insulin       4109        0.077253
                   is_pregnant          0        0.000000
subject_split_across_traintest          0        0.000000
                     ethnicity          0        0.000000
              age_of_diagnosis          0        0.000000
                        gender          0        0.000000
                        weight          0        0.000000
                   source_file          0        0.000000
                       is_test          0        0.000000
     insulin_delivery_modality          0        0.000000
            insulin_type_bolus          0        0

In [2]:
import pyarrow.parquet as pq
import pyarrow as pa

input_path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAP.parquet'

output_path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAPdropped.parquet'
# Columns to drop
drop_cols = [
    "is_pregnant",
    "ethnicity",
    "insulin_delivery_device",
    "cgm_device",
    "is_test",
    "source_file",
    "subject_split_across_traintest",
    "patient_stream_id",
]

pf = pq.ParquetFile(input_path)

# Get remaining columns
remaining_cols = [
    col for col in pf.schema.names
    if col not in drop_cols
]

print("Columns being kept:")
for col in remaining_cols:
    print(" -", col)

print("\nColumns being dropped:")
for col in drop_cols:
    print(" -", col)

# Create writer using the original schema minus dropped columns
first_table = pf.read_row_group(0, columns=remaining_cols)

writer = pq.ParquetWriter(
    output_path,
    first_table.schema,
    compression="snappy"
)

# Write first row group
writer.write_table(first_table)

# Write remaining row groups
for i in range(1, pf.num_row_groups):
    table = pf.read_row_group(i, columns=remaining_cols)
    writer.write_table(table)

    print(f"Processed row group {i + 1}/{pf.num_row_groups}")

writer.close()

print("\nDone!")
print(f"Saved to:\n{output_path}")

Columns being kept:
 - CGM
 - age
 - basal
 - bolus
 - carbs
 - date
 - height
 - id
 - insulin
 - insulin_delivery_algorithm
 - insulin_delivery_modality
 - insulin_type_basal
 - insulin_type_bolus
 - weight
 - gender
 - age_of_diagnosis

Columns being dropped:
 - is_pregnant
 - ethnicity
 - insulin_delivery_device
 - cgm_device
 - is_test
 - source_file
 - subject_split_across_traintest
 - patient_stream_id
Processed row group 2/56
Processed row group 3/56
Processed row group 4/56
Processed row group 5/56
Processed row group 6/56
Processed row group 7/56
Processed row group 8/56
Processed row group 9/56
Processed row group 10/56
Processed row group 11/56
Processed row group 12/56
Processed row group 13/56
Processed row group 14/56
Processed row group 15/56
Processed row group 16/56
Processed row group 17/56
Processed row group 18/56
Processed row group 19/56
Processed row group 20/56
Processed row group 21/56
Processed row group 22/56
Processed row group 23/56
Processed row group 24/

In [3]:
import pyarrow.parquet as pq
import pandas as pd

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAPdropped.parquet'

pf = pq.ParquetFile(path)
total_rows = pf.metadata.num_rows

counts = {
    "CGM available": 0,
    "Carbs available": 0,
    "Bolus available": 0,
    "CGM + Carbs available": 0,
    "CGM + Bolus available": 0,
    "Carbs + Bolus available": 0,
    "CGM + Carbs + Bolus available": 0,
}

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["CGM", "carbs", "bolus"]
):
    df = batch.to_pandas()

    cgm = df["CGM"].notna()
    carbs = df["carbs"].notna()
    bolus = df["bolus"].notna()

    counts["CGM available"] += cgm.sum()
    counts["Carbs available"] += carbs.sum()
    counts["Bolus available"] += bolus.sum()

    counts["CGM + Carbs available"] += (cgm & carbs).sum()
    counts["CGM + Bolus available"] += (cgm & bolus).sum()
    counts["Carbs + Bolus available"] += (carbs & bolus).sum()
    counts["CGM + Carbs + Bolus available"] += (cgm & carbs & bolus).sum()

print(f"Total rows: {total_rows:,}")
print("=" * 80)

for name, count in counts.items():
    percentage = count / total_rows * 100
    print(f"{name:<35} {count:>15,}  ({percentage:6.2f}%)")

Total rows: 5,318,918
CGM available                             4,815,456  ( 90.53%)
Carbs available                           5,309,931  ( 99.83%)
Bolus available                           5,313,412  ( 99.90%)
CGM + Carbs available                     4,807,090  ( 90.38%)
CGM + Bolus available                     4,810,371  ( 90.44%)
Carbs + Bolus available                   5,309,928  ( 99.83%)
CGM + Carbs + Bolus available             4,807,087  ( 90.38%)


In [4]:
import pyarrow.parquet as pq
import pandas as pd

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/PEDAPdropped.parquet'

pf = pq.ParquetFile(path)
total_rows = pf.metadata.num_rows

counts = {
    "carbs_nan": 0,
    "carbs_zero": 0,
    "carbs_positive": 0,
    "bolus_nan": 0,
    "bolus_zero": 0,
    "bolus_positive": 0,
    
    "bolus_zero_insulin_equals_basal": 0,
    "bolus_zero_with_values": 0,
    
    "bolus_positive_insulin_matches_sum": 0,
    "bolus_positive_with_values": 0,
}

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["carbs", "bolus", "basal", "insulin"]
):
    df = batch.to_pandas()

    # -------------------------
    # CARBS
    # -------------------------
    carbs_nan = df["carbs"].isna()
    carbs_zero = df["carbs"].notna() & (df["carbs"] == 0)
    carbs_positive = df["carbs"] > 0

    counts["carbs_nan"] += carbs_nan.sum()
    counts["carbs_zero"] += carbs_zero.sum()
    counts["carbs_positive"] += carbs_positive.sum()

    # -------------------------
    # BOLUS
    # -------------------------
    bolus_nan = df["bolus"].isna()
    bolus_zero = df["bolus"].notna() & (df["bolus"] == 0)
    bolus_positive = df["bolus"] > 0

    counts["bolus_nan"] += bolus_nan.sum()
    counts["bolus_zero"] += bolus_zero.sum()
    counts["bolus_positive"] += bolus_positive.sum()

    # -------------------------
    # BOLUS = 0
    # Check insulin ≈ basal
    # -------------------------
    valid_zero = (
        bolus_zero &
        df["insulin"].notna() &
        df["basal"].notna()
    )

    counts["bolus_zero_with_values"] += valid_zero.sum()

    counts["bolus_zero_insulin_equals_basal"] += (
        valid_zero &
        ((df["insulin"] - df["basal"]).abs() < 1e-6)
    ).sum()

    # -------------------------
    # BOLUS > 0
    # Check insulin ≈ basal + bolus
    # -------------------------
    valid_positive = (
        bolus_positive &
        df["insulin"].notna() &
        df["basal"].notna()
    )

    counts["bolus_positive_with_values"] += valid_positive.sum()

    counts["bolus_positive_insulin_matches_sum"] += (
        valid_positive &
        ((df["insulin"] - (df["basal"] + df["bolus"])).abs() < 1e-6)
    ).sum()


# ============================================================
# PRINT RESULTS
# ============================================================

print(f"Total rows: {total_rows:,}")
print("=" * 90)

print("\nCARBS")
print("-" * 90)

for key in ["carbs_nan", "carbs_zero", "carbs_positive"]:
    count = counts[key]
    print(f"{key:<25} {count:>15,}  ({count / total_rows * 100:6.2f}%)")


print("\nBOLUS")
print("-" * 90)

for key in ["bolus_nan", "bolus_zero", "bolus_positive"]:
    count = counts[key]
    print(f"{key:<25} {count:>15,}  ({count / total_rows * 100:6.2f}%)")


print("\nBOLUS = 0 → INSULIN vs BASAL")
print("-" * 90)

print(
    f"Rows with bolus=0 and insulin/basal available: "
    f"{counts['bolus_zero_with_values']:,}"
)

print(
    f"Rows where insulin ≈ basal: "
    f"{counts['bolus_zero_insulin_equals_basal']:,}"
)

if counts["bolus_zero_with_values"] > 0:
    pct = (
        counts["bolus_zero_insulin_equals_basal"]
        / counts["bolus_zero_with_values"]
        * 100
    )
    print(f"Percentage: {pct:.2f}%")


print("\nBOLUS > 0 → INSULIN vs BASAL + BOLUS")
print("-" * 90)

print(
    f"Rows with bolus>0 and insulin/basal available: "
    f"{counts['bolus_positive_with_values']:,}"
)

print(
    f"Rows where insulin ≈ basal + bolus: "
    f"{counts['bolus_positive_insulin_matches_sum']:,}"
)

if counts["bolus_positive_with_values"] > 0:
    pct = (
        counts["bolus_positive_insulin_matches_sum"]
        / counts["bolus_positive_with_values"]
        * 100
    )
    print(f"Percentage: {pct:.2f}%")

Total rows: 5,318,918

CARBS
------------------------------------------------------------------------------------------
carbs_nan                           8,987  (  0.17%)
carbs_zero                      5,210,983  ( 97.97%)
carbs_positive                     98,948  (  1.86%)

BOLUS
------------------------------------------------------------------------------------------
bolus_nan                           5,506  (  0.10%)
bolus_zero                      4,944,871  ( 92.97%)
bolus_positive                    368,541  (  6.93%)

BOLUS = 0 → INSULIN vs BASAL
------------------------------------------------------------------------------------------
Rows with bolus=0 and insulin/basal available: 2,714,637
Rows where insulin ≈ basal: 2,714,637
Percentage: 100.00%

BOLUS > 0 → INSULIN vs BASAL + BOLUS
------------------------------------------------------------------------------------------
Rows with bolus>0 and insulin/basal available: 179,700
Rows where insulin ≈ basal + bolus: 179,700
